# Periodic FD–OPT systematic benchmarks

Boundary-free comparisons using the shared famous-equation benchmark matrix. Raw weak-residual orders include the cell-volume factor $h^d$.

In [ ]:
using CairoMakie, JLD2, Statistics, LinearAlgebra

root = normpath(joinpath(@__DIR__, ".."))
datadir = joinpath(root, "scripts", "tmp", "famous_equation_periodic_benchmarks")
construction_dir = joinpath(root, "scripts", "tmp", "famous_equation_interior_diagnostics")

poisson = Dict(
    1 => load(joinpath(datadir, "poisson_fd_vs_opt_1d.jld2")),
    2 => load(joinpath(datadir, "poisson_fd_vs_opt_2d.jld2")),
)
scheme_names(d) = getproperty.(d["schemes"], :name)
case_names(d) = getproperty.(d["cases"], :name)
nothing

## Absolute and relative residual convergence

In [ ]:
fig_convergence = Figure(size=(1200, 850))
for (column, dimension) in enumerate((1, 2))
    d = poisson[dimension]
    r = d["results"]["dimension_$(dimension)"]
    names = scheme_names(d)
    dx = 2π ./ r.sizes
    homogeneous = findfirst(==("homogeneous"), case_names(d))
    for (row, metric) in enumerate((:absolute_errors, :relative_errors))
        ax = Axis(fig_convergence[row, column], xscale=log10, yscale=log10,
            xlabel="Δx", ylabel=string(metric),
            title="Poisson $(dimension)D — $(replace(string(metric), "_" => " "))")
        values = getproperty(r, metric)
        for scheme in eachindex(names)
            lines!(ax, dx, values[:, homogeneous, scheme], label=names[scheme])
            scatter!(ax, dx, values[:, homogeneous, scheme])
        end
        axislegend(ax, position=:lt)
    end
end
fig_convergence

## Observed differential-operator order by material case

In [ ]:
fig_orders = Figure(size=(1250, 700))
for (column, dimension) in enumerate((1, 2))
    d = poisson[dimension]
    r = d["results"]["dimension_$(dimension)"]
    names = scheme_names(d)
    cases = case_names(d)
    # Remove the weak cell-volume contribution h^dimension.
    order_matrix = [median(filter(isfinite, r.relative_orders[:, c, s])) - dimension
        for c in eachindex(cases), s in eachindex(names)]
    ax = Axis(fig_orders[1, column], xticks=(eachindex(names), names),
        yticks=(eachindex(cases), cases), title="Poisson $(dimension)D predicted PDE order",
        xticklabelrotation=π/4)
    hm = heatmap!(ax, order_matrix, colorrange=(0, 6))
    Colorbar(fig_orders[2, column], hm, vertical=false, label="observed order")
end
fig_orders

## OPT/FD absolute-error ratios

In [ ]:
fig_ratios = Figure(size=(1200, 500))
for (column, dimension) in enumerate((1, 2))
    d = poisson[dimension]
    r = d["results"]["dimension_$(dimension)"]
    dx = 2π ./ r.sizes
    cases = case_names(d)
    ax = Axis(fig_ratios[1, column], xscale=log10, yscale=log10,
        xlabel="Δx", ylabel="OPT residual / FD residual",
        title="Poisson $(dimension)D")
    for case in eachindex(cases)
        lines!(ax, dx, r.relative_errors[:, case, 2] ./ r.relative_errors[:, case, 1],
            label="OPT3/FD3 — $(cases[case])")
        lines!(ax, dx, r.relative_errors[:, case, 4] ./ r.relative_errors[:, case, 3],
            linestyle=:dash, label="OPT5/FD5 — $(cases[case])")
    end
    hlines!(ax, [1.0], color=:black, linestyle=:dot)
    axislegend(ax, position=:lt, labelsize=9)
end
fig_ratios

## CPU time: recette construction and residual application

In [ ]:
fig_cpu = Figure(size=(1200, 780))
for (column, dimension) in enumerate((1, 2))
    d = poisson[dimension]
    r = d["results"]["dimension_$(dimension)"]
    names = scheme_names(d)
    ax1 = Axis(fig_cpu[1, column], xticks=(eachindex(names), names),
        ylabel="seconds", title="Poisson $(dimension)D recipe construction",
        xticklabelrotation=π/4)
    barplot!(ax1, eachindex(names), [median(r.recipe_seconds[:, s]) for s in eachindex(names)])
    ax2 = Axis(fig_cpu[2, column], xscale=log10, yscale=log10,
        xlabel="number of spatial nodes", ylabel="seconds / material case",
        title="Poisson $(dimension)D residual evaluation")
    for s in eachindex(names)
        lines!(ax2, r.sizes .^ dimension,
            [median(r.residual_seconds[g, :, s]) for g in eachindex(r.sizes)], label=names[s])
    end
    axislegend(ax2, position=:lt)
end
fig_cpu

## Wave construction cost and support size

In [ ]:
construction_files = filter(isfile, [
    joinpath(construction_dir, "recipe_construction_waves_max2d_time3_default.jld2"),
    joinpath(construction_dir, "recipe_construction_all_max2d_time3_default.jld2"),
])
construction_rows = reduce(vcat, [load(file)["rows"] for file in construction_files]; init=NamedTuple[])
unique_rows = Dict((row.equation, row.recipe) => row for row in construction_rows)
wave_rows = collect(values(unique_rows))
fig_wave_cost = Figure(size=(1150, 500))
labels = ["$(row.equation_label)\n$(row.recipe)" for row in wave_rows]
ax_construction = Axis(fig_wave_cost[1, 1], xticks=(eachindex(labels), labels),
    xticklabelrotation=π/3, ylabel="seconds", title="Cached/compiled recipe construction")
barplot!(ax_construction, eachindex(labels), getproperty.(wave_rows, :build_seconds))
ax2 = Axis(fig_wave_cost[1, 2], xticks=(eachindex(labels), labels),
    xticklabelrotation=π/3, ylabel="nonzero Γ coefficients", title="Source redistribution support")
barplot!(ax2, eachindex(labels), getproperty.(wave_rows, :gamma_nonzero))
fig_wave_cost

## Wave manufactured-solution convergence and absolute error

Every Δ uses an independently constructed recette. Errors are PDE-equivalent RMS residuals, `(A*u-Γ*f)/sum(abs(Γ))`; the source redistribution is therefore included.

In [ ]:
wave_file = joinpath(datadir, "wave_fd_vs_opt.jld2")
wave = load(wave_file)
wave_convergence = wave["convergence_rows"]
wave_timing = wave["timing_rows"]
wave_equations = unique(getproperty.(wave_convergence, :equation))
wave_schemes = unique(getproperty.(wave_convergence, :scheme))

fig_wave_convergence = Figure(size=(1250, 330 * length(wave_equations)))
for (panel, equation) in enumerate(wave_equations)
    ax_wave = Axis(fig_wave_convergence[panel, 1], xscale=log10, yscale=log10,
        xlabel="Δx", ylabel="absolute PDE residual",
        title="$equation — homogeneous manufactured field")
    equation_rows = filter(r -> r.equation == equation && r.scenario == "homogeneous",
        wave_convergence)
    for scheme in wave_schemes, branch in unique(getproperty.(equation_rows, :branch))
        selected = sort(filter(r -> r.scheme == scheme && r.branch == branch,
            equation_rows); by=r -> r.dx, rev=true)
        isempty(selected) && continue
        label = branch in ("SH", "P") && length(unique(getproperty.(equation_rows, :branch))) == 1 ?
            scheme : "$scheme — $branch"
        lines!(ax_wave, getproperty.(selected, :dx), getproperty.(selected, :absolute_error),
            label=label)
        scatter!(ax_wave, getproperty.(selected, :dx), getproperty.(selected, :absolute_error))
    end
    axislegend(ax_wave, position=:lt, labelsize=9)
end
fig_wave_convergence

## Wave observed order and fine-grid material sensitivity

In [ ]:
fig_wave_summary = Figure(size=(1250, 350 * length(wave_equations)))
for (panel, equation) in enumerate(wave_equations)
    equation_rows = filter(r -> r.equation == equation, wave_convergence)
    order_values = [median(filter(isfinite,
        getproperty.(filter(r -> r.scheme == scheme, equation_rows), :observed_order)))
        for scheme in wave_schemes]
    ax_order = Axis(fig_wave_summary[panel, 1],
        xticks=(eachindex(wave_schemes), wave_schemes), xticklabelrotation=π/4,
        ylabel="median observed order", title="$equation — all materials/branches")
    barplot!(ax_order, eachindex(wave_schemes), order_values)
    hlines!(ax_order, [0.0], color=:gray, linestyle=:dot)

    fine_n = maximum(getproperty.(equation_rows, :n))
    fine = filter(r -> r.n == fine_n, equation_rows)
    scenarios = unique(getproperty.(fine, :scenario))
    sensitivity = [log10(median(getproperty.(filter(r ->
        r.scheme == scheme && r.scenario == scenario, fine), :absolute_error)))
        for scenario in scenarios, scheme in wave_schemes]
    ax_error = Axis(fig_wave_summary[panel, 2],
        xticks=(eachindex(wave_schemes), wave_schemes),
        yticks=(eachindex(scenarios), scenarios), xticklabelrotation=π/4,
        title="$equation — log₁₀ fine-grid absolute error")
    hm = heatmap!(ax_error, sensitivity)
    Colorbar(fig_wave_summary[panel, 3], hm, label="log₁₀ error")
end
fig_wave_summary

## Wave OPT/FD ratios and CPU cost

In [ ]:
scheme_pairs = [("OPT3", "FD3"),
    ("OPT5space-OPT3time-hat-supp0", "FD5space-FD3time")]
fig_wave_ratio_cpu = Figure(size=(1250, 350 * length(wave_equations)))
for (panel, equation) in enumerate(wave_equations)
    equation_rows = filter(r -> r.equation == equation, wave_convergence)
    ax_ratio = Axis(fig_wave_ratio_cpu[panel, 1], xscale=log10, yscale=log10,
        xlabel="Δx", ylabel="median OPT/FD absolute error", title="$equation — all cases")
    for (opt, fd) in scheme_pairs
        dx_values = sort(unique(getproperty.(filter(r -> r.scheme == opt, equation_rows), :dx)); rev=true)
        ratios = [median(getproperty.(filter(r -> r.scheme == opt && r.dx == dx,
            equation_rows), :absolute_error)) /
            median(getproperty.(filter(r -> r.scheme == fd && r.dx == dx,
            equation_rows), :absolute_error)) for dx in dx_values]
        lines!(ax_ratio, dx_values, ratios, label="$opt / $fd")
        scatter!(ax_ratio, dx_values, ratios)
    end
    hlines!(ax_ratio, [1.0], color=:gray, linestyle=:dot)
    axislegend(ax_ratio, labelsize=8)

    timing = filter(r -> r.equation == equation, wave_timing)
    ax_cpu = Axis(fig_wave_ratio_cpu[panel, 2], yscale=log10,
        xticks=(eachindex(wave_schemes), wave_schemes), xticklabelrotation=π/4,
        ylabel="seconds", title="$equation — median CPU")
    construction = [median(getproperty.(filter(r -> r.scheme == scheme, timing), :recipe_seconds))
        for scheme in wave_schemes]
    application = [median(getproperty.(filter(r -> r.scheme == scheme, timing),
        :symbol_application_seconds)) for scheme in wave_schemes]
    barplot!(ax_cpu, eachindex(wave_schemes) .- 0.18, construction,
        width=0.34, label="recette construction")
    barplot!(ax_cpu, eachindex(wave_schemes) .+ 0.18, application,
        width=0.34, label="local symbol application")
    axislegend(ax_cpu, labelsize=8)
end
fig_wave_ratio_cpu

## CFL, dispersion and elastic P/S branches

CFL is obtained from the homogeneous amplification roots. Elastic P/S curves are retained as diagnostics even though the manufactured elastic residual currently fails the convergence check.

In [ ]:
if isfile(wave_file)
    fig_cfl = Figure(size=(1100, 450))
    ax = Axis(fig_cfl[1, 1], ylabel="maximum stable CFL", title="CFL by PDE and scheme")
    rows = wave["cfl_rows"]
    labels = ["$(r.equation) $(r.scheme) $(r.branch)" for r in rows]
    ax.xticks = (eachindex(labels), labels)
    ax.xticklabelrotation = π/3
    barplot!(ax, eachindex(rows), getproperty.(rows, :cfl_max))
    fig_cfl
else
    @info "wave_fd_vs_opt.jld2 not yet present; CFL figure deferred"
end

In [ ]:
if isfile(wave_file)
    wave = load(wave_file)
    rows = wave["dispersion_rows"]
    groups = unique((r.equation, r.branch, r.direction) for r in rows)
    fig_dispersion = Figure(size=(1200, 350 * length(groups)))
    for (panel, group) in enumerate(groups)
        ax_dispersion = Axis(fig_dispersion[panel, 1], xlabel="points per wavelength",
            ylabel="numerical / exact phase velocity − 1",
            title="$(group[1]) — $(group[2]) branch — $(group[3])")
        for scheme in unique(r.scheme for r in rows if (r.equation, r.branch, r.direction) == group)
            selected = sort(filter(r -> (r.equation, r.branch, r.direction) == group &&
                r.scheme == scheme, rows); by=r -> r.points_per_wavelength)
            lines!(ax_dispersion, getproperty.(selected, :points_per_wavelength),
                getproperty.(selected, :relative_phase_error), label=scheme)
        end
        axislegend(ax_dispersion)
    end
    fig_dispersion
else
    @info "wave_fd_vs_opt.jld2 not yet present; dispersion/P-S figures deferred"
end